# Chest X-ray Classification
A university project classifying COVID, Normal and Viral Pneumonia radiographs using a CNN trained from scratch. The computational protocol is preserved from TASK1.ipynb. Historical results below were saved in the original Colab run; training has not been repeated for this release.

## Dataset
The original run contains 9,961 images: COVID 3,616; Normal 5,000; Viral Pneumonia 1,345. This is a local subset, not a claim about the size of the full public dataset. The selection of the 5,000 Normal images is not documented in the source.

Source: [COVID-19 Radiography Database](https://www.kaggle.com/datasets/tawsifurrahman/covid19-radiography-database). Obtain the data separately and follow its terms and citation requirements. No radiographs are distributed here.

## Dependencies and reproducibility
Install `requirements.txt` in a Python 3.10 or 3.11 environment. The proposed dependency ranges are not the recovered original environment. Random seeds are preserved; hardware, library versions, file enumeration order and unseeded augmentation can affect results. Mixed precision is retained from Colab; a GPU is recommended.


In [ ]:
import os
# Optional in Colab: uncomment these lines to mount Google Drive.
# from google.colab import drive
# drive.mount('/content/drive')
DATASET_ROOT = os.environ.get('CHEST_XRAY_DATASET', 'data/COVID-19_Radiography_Dataset')
CLASS_NAMES = ['COVID', 'Normal', 'Viral Pneumonia']
print('Dataset root:', DATASET_ROOT)


## Dependencies and random seeds


In [ ]:
import os, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from glob import glob
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score
from sklearn.utils.class_weight import compute_class_weight
import seaborn as sns
from tensorflow.keras.optimizers import AdamW


np.random.seed(42)
tf.random.set_seed(42)

tf.keras.mixed_precision.set_global_policy("mixed_float16")
print("GPU devices:", tf.config.list_physical_devices('GPU'))

## Data loading and class distribution


In [ ]:
def load_dataset_df(base_path, class_names): #Scans the dataset directories and creates a DataFrame containing file paths and labels
    data = []
    for cls in class_names:
        img_dir = os.path.join(base_path, cls, "images")
        files = []
        files += glob(os.path.join(img_dir, "*.png"))
        files += glob(os.path.join(img_dir, "*.jpg"))
        files += glob(os.path.join(img_dir, "*.jpeg"))
        for f in files:
            data.append({"filepath": f, "label": cls})
    return pd.DataFrame(data)

df = load_dataset_df(DATASET_ROOT, CLASS_NAMES)

if df.empty:
    raise FileNotFoundError("No images found: check DATASET_ROOT and CLASS/images directories.")
print("Total images:", len(df))
print("\nImages per class:")
print(df["label"].value_counts().to_string())

le = LabelEncoder()
df["label_encoded"] = le.fit_transform(df["label"])

# Store the real order used by the LabelEncoder
class_names = list(le.classs_)
print("\nClass mapping (LabelEncoder):")
for i, c in enumerate(class_names):
    print(f"  {c} -> {i}")

X_paths = df["filepath"].values
y = df["label_encoded"].values

# Display the first few rows of the processed DataFrame
df.head()

## Image preprocessing
Images are decoded as grayscale, resized to 224 × 224 and divided by 255. Contrast factor 1.15 is applied in make_dataset to all splits. The optional preview in the original notebook precedes contrast adjustment and is omitted to avoid distributing a radiograph.

In [ ]:
IMG_SIZE = (224, 224)
AUTOTUNE = tf.data.AUTOTUNE

# Apply stable and light contrast boost
def contrast_boost_tf(img):
    img = tf.image.adjust_contrast(img, 1.15)
    img = tf.clip_by_value(img, 0.0, 1.0)
    return img

def preprocess_gray(path, label):
    img_bytes = tf.io.read_file(path)
    img = tf.image.decode_image(img_bytes, channels=1, expand_animations=False) # Decode as grayscale 1 channel
    img = tf.image.resize(img, IMG_SIZE) # Resize
    img = tf.cast(img, tf.float32) / 255.0 # Normalize pixel values to 0-1
    return img, label

## Training augmentation and input pipeline
Only training receives horizontal flips, rotation factor 0.03 (±10.8 degrees, correcting the original ~5 degree comment) and zoom 0.05. Validation and test use deterministic preprocessing.

In [ ]:
data_augmentation = tf.keras.Sequential(
    [
        tf.keras.layers.RandomFlip("horizontal"),
        tf.keras.layers.RandomRotation(0.03),   # ±10.8 degrees
        tf.keras.layers.RandomZoom(0.05),
    ],
    name="augmentations"
)

def make_dataset(paths, labels, batch_size, training):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    if training:
        ds = ds.shuffle(len(paths), seed=42, reshuffle_each_iteration=True)

    # Preprocessing
    ds = ds.map(preprocess_gray, num_parallel_calls=AUTOTUNE)

    # Apply deterministic contrast enhancement to all sets
    ds = ds.map(lambda x, y: (contrast_boost_tf(x), y),
                num_parallel_calls=AUTOTUNE)

    # Apply data augmentation only to the training set
    if training:
        ds = ds.map(lambda x, y: (data_augmentation(x, training=True), y),
                    num_parallel_calls=AUTOTUNE)

    ds = ds.batch(batch_size).prefetch(AUTOTUNE)
    return ds

## CNN architecture
Four blocks use 32, 64, 128 and 256 filters, 3 × 3 convolutions without bias, batch normalization, ReLU and max pooling. Block dropout is 0, 0.1, 0.2 and 0.3; global average pooling precedes dropout 0.4 and a float32 three-class softmax. AdamW uses weight decay 1e-5.

In [ ]:
def build_cnn(lr, n_classs):
    inputs = tf.keras.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 1))

    def block(x, filters, drop=0.0):
      # Convolutional layer with Batch Normalization and ReLU activation
        x = tf.keras.layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.ReLU()(x)
        x = tf.keras.layers.MaxPool2D()(x)
        if drop > 0:
            x = tf.keras.layers.Dropout(drop)(x)
        return x

    # Increasing filter sizes
    x = block(inputs, 32, 0.0)
    x = block(x, 64,  0.1)
    x = block(x, 128, 0.2)
    x = block(x, 256, 0.3)

    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.4)(x)

    outputs = tf.keras.layers.Dense(n_classs, activation="softmax", dtype="float32")(x)

    model = tf.keras.Model(inputs, outputs)
    model.compile(
        optimizer=AdamW(learning_rate=lr, weight_decay=1e-5),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

## Nested cross-validation and hyperparameter search
Three stratified outer folds (seed 42) estimate performance. Three inner folds (seed 123) select learning rate {3e-4, 1e-4} and batch size {16, 32} by mean macro F1. Outer test labels are not used in tuning.

In [ ]:
# Setup Nested Cross-Validation folds
outer_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=123)

param_grid = [
    {"lr": 3e-4, "batch": 16},
    {"lr": 1e-4, "batch": 16},
    {"lr": 3e-4, "batch": 32},
    {"lr": 1e-4, "batch": 32},
]

def macro_metrics(y_true, y_pred):
    p = precision_score(y_true, y_pred, average="macro", zero_division=0)
    r = recall_score(y_true, y_pred, average="macro", zero_division=0)
    f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    return float(p), float(r), float(f1)

## Outer-fold class distributions


In [ ]:
for outer_i, (train_idx, test_idx) in enumerate(outer_cv.split(X_paths, y)):
    print(f"Outer Fold {outer_i}:")
    print("  Training distribution:", np.bincount(y[train_idx]))
    print("  Test distribution: ", np.bincount(y[test_idx]))

## Model training and outer-test evaluation
Each inner fit has at most 7 epochs, early-stopping patience 3 and learning-rate reduction patience 1. Final fits use a stratified 80/20 split of the outer-training partition (seed 7), at most 25 epochs, early-stopping patience 5 and reduction patience 2. Both restore the best validation-loss weights; learning-rate reduction uses factor 0.5 and minimum 1e-6. Class weights are calculated from the actual training subset only. Inner validation also controls stopping and learning-rate scheduling; its scores are selection scores, not unbiased performance estimates.

In [ ]:
outer_results = []
outer_best_params = []
histories_outer = {}
all_reports_list = []
y_test_all = []
y_pred_all = []
fold_id_all = []

outer_fold_labels = ["A", "B", "C"]

# Start Outer Loop for model evaluation
for outer_i, (train_idx, test_idx) in enumerate(outer_cv.split(X_paths, y)):
    fold_name = outer_fold_labels[outer_i]
    print("\n" + "=" * 80)
    print(f"OUTER FOLD {fold_name}")
    print("=" * 80)

    X_train_outer, X_test_outer = X_paths[train_idx], X_paths[test_idx]
    y_train_outer, y_test_outer = y[train_idx], y[test_idx]


    # Inner Loop for hyperparameter tuning
    best_mean_f1 = -1.0
    best_params = None

    for params in param_grid:
        inner_f1s = []

        for tr_i, va_i in inner_cv.split(X_train_outer, y_train_outer):
            X_tr, X_va = X_train_outer[tr_i], X_train_outer[va_i]
            y_tr, y_va = y_train_outer[tr_i], y_train_outer[va_i]

            tr_ds = make_dataset(X_tr, y_tr, batch_size=params["batch"], training=True)
            va_ds = make_dataset(X_va, y_va, batch_size=params["batch"], training=False)

            tf.keras.backend.clear_session()
            model = build_cnn(lr=params["lr"], n_classs=len(class_names))

            # Class weights (inner loop)
            classs = np.unique(y_tr)
            cw = compute_class_weight(class_weight="balanced", classs=classs, y=y_tr)
            class_weight_inner = {int(c): float(w) for c, w in zip(classs, cw)}

            es = tf.keras.callbacks.EarlyStopping(
                monitor="val_loss", patience=3, restore_best_weights=True, verbose=0
            )

            rlr_inner = tf.keras.callbacks.ReduceLROnPlateau(
                      monitor="val_loss", factor=0.5, patience=1, min_lr=1e-6, verbose=0
            )

            model.fit(
                tr_ds,
                validation_data=va_ds,
                epochs=7,   # inner training epoch limit
                verbose=0,
                callbacks=[es,rlr_inner],
                class_weight=class_weight_inner
            )
            # Inner model performance
            y_va_pred = np.argmax(model.predict(va_ds, verbose=0), axis=1)
            _, _, f1m = macro_metrics(y_va, y_va_pred)
            inner_f1s.append(f1m)

            tf.keras.backend.clear_session()
            gc.collect()

        mean_f1 = float(np.mean(inner_f1s))
        print(f"params={params} -> mean inner MacroF1={mean_f1:.4f}")

        if mean_f1 > best_mean_f1:
            best_mean_f1 = mean_f1
            best_params = params

    print(f">> BEST OUTER {fold_name}: {best_params} (mean inner MacroF1={best_mean_f1:.4f})")
    outer_best_params.append((fold_name, best_params, best_mean_f1))

    # Final fit on 80% of outer training; reserve 20% for validation
    X_tr_final, X_va_final, y_tr_final, y_va_final = train_test_split(
        X_train_outer, y_train_outer, test_size=0.2, random_state=7, stratify=y_train_outer
    )

    tr_ds_final = make_dataset(X_tr_final, y_tr_final, batch_size=best_params["batch"], training=True)
    va_ds_final = make_dataset(X_va_final, y_va_final, batch_size=best_params["batch"], training=False)
    te_ds       = make_dataset(X_test_outer, y_test_outer, batch_size=best_params["batch"], training=False)

    # Class weights (final training)
    classs = np.unique(y_tr_final)
    cw = compute_class_weight(class_weight="balanced", classs=classs, y=y_tr_final)
    class_weight_final = {int(c): float(w) for c, w in zip(classs, cw)}

    tf.keras.backend.clear_session()
    model = build_cnn(lr=best_params["lr"], n_classs=len(class_names))

    es_final = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1
    )

    rlr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    min_lr=1e-6,
    verbose=1
    )


    history = model.fit(
        tr_ds_final,
        validation_data=va_ds_final,
        epochs=25,
        verbose=2,
        callbacks=[es_final,rlr],
        class_weight=class_weight_final
    )

    histories_outer[fold_name] = history.history

    # Final test on outer test set
    y_test_pred = np.argmax(model.predict(te_ds, verbose=0), axis=1)

    y_test_all.append(y_test_outer)
    y_pred_all.append(y_test_pred)
    fold_id_all.append(np.full_like(y_test_pred, outer_i))  # optional

    p, r, f1m = macro_metrics(y_test_outer, y_test_pred)
    outer_results.append((p, r, f1m))
    print(f"\nOUTER TEST {fold_name}: MacroP={p:.4f} MacroR={r:.4f} MacroF1={f1m:.4f}")

    print("\nCLASSIFICATION REPORT (OUTER TEST):")
    print(classification_report(
        y_test_outer, y_test_pred, target_names=class_names, digits=4, zero_division=0
    ))

    rep = classification_report(
        y_test_outer, y_test_pred, target_names=class_names,
        output_dict=True, zero_division=0
    )
    rep_df = pd.DataFrame(rep).T.reset_index().rename(columns={"index": "class"})
    rep_df["fold"] = fold_name
    all_reports_list.append(rep_df)

    # Plot Confusion Matrix for the current fold
    cm = confusion_matrix(y_test_outer, y_test_pred)
    plt.figure(figsize=(6,5))
    sns.heatmap(cm, annot=True, fmt="d",
                xticklabels=class_names, yticklabels=class_names, cmap="Blues")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(f"Confusion Matrix – Outer Fold {fold_name}")
    plt.show()

    tf.keras.backend.clear_session()
    gc.collect()

## Training and validation curves


In [ ]:
def plot_history(history_dict, fold_name):
    epochs_range = range(1, len(history_dict["loss"]) + 1)

    fig, ax = plt.subplots(1, 2, figsize=(18, 6))

    # Accuracy
    ax[0].plot(epochs_range, history_dict["accuracy"], label="Train Acc")
    ax[0].plot(epochs_range, history_dict["val_accuracy"], label="Val Acc")
    ax[0].set_title(f"Accuracy – Outer Fold {fold_name}")
    ax[0].set_xlabel("Epochs")
    ax[0].set_ylabel("Accuracy")
    ax[0].legend()

    # Loss
    ax[1].plot(epochs_range, history_dict["loss"], label="Train Loss")
    ax[1].plot(epochs_range, history_dict["val_loss"], label="Val Loss")
    ax[1].set_title(f"Loss – Outer Fold {fold_name}")
    ax[1].set_xlabel("Epochs")
    ax[1].set_ylabel("Loss")
    ax[1].legend()

    plt.show()

print("Available histories:", list(histories_outer.keys()))
for fold_name, h in histories_outer.items():
    plot_history(h, fold_name)

## Aggregated outer-test predictions


In [ ]:
y_test_all_concat = np.concatenate(y_test_all, axis=0)
y_pred_all_concat = np.concatenate(y_pred_all, axis=0)

print("Total pooled outer-test samples:", len(y_test_all_concat))

# Report for all combined folds
print("\nPOOLED CLASSIFICATION REPORT (A+B+C):")
print(classification_report(
    y_test_all_concat,
    y_pred_all_concat,
    target_names=class_names,
    digits=4,
    zero_division=0
))

# Aggregated confusion matrix
cm_global = confusion_matrix(y_test_all_concat, y_pred_all_concat)

plt.figure(figsize=(7,6))
sns.heatmap(
    cm_global,
    annot=True,
    fmt="d",
    xticklabels=class_names,
    yticklabels=class_names,
    cmap="Blues"
)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Aggregated Confusion Matrix – Outer Test (A+B+C)")
plt.show()

## Metrics for each outer fold


In [ ]:
df_folds = pd.DataFrame(
    outer_results,
    columns=["Macro Precision", "Macro Recall", "Macro F1"],
    index=["Fold A", "Fold B", "Fold C"]
)

display(df_folds)

## Mean and sample standard deviation across outer folds
Standard deviation uses ddof=1 over three folds. It is not a confidence interval. Pooled metrics differ from the arithmetic mean of fold metrics.

In [ ]:
outer_results = np.array(outer_results)

print("\n" + "#" * 80)
print("FINAL RESULTS — NESTED 3×3 (outer-test mean ± sample SD)")
print("#" * 80)
print(f"Macro Precision: {outer_results[:,0].mean():.4f} ± {outer_results[:,0].std(ddof=1):.4f}")
print(f"Macro Recall:    {outer_results[:,1].mean():.4f} ± {outer_results[:,1].std(ddof=1):.4f}")
print(f"Macro F1:        {outer_results[:,2].mean():.4f} ± {outer_results[:,2].std(ddof=1):.4f}")
print("\nBest params per fold:", outer_best_params)

## Per-class mean and sample standard deviation


In [ ]:
df_all_reports = pd.concat(all_reports_list, ignore_index=True)
real_classs = list(class_names)

df_classs_only = df_all_reports[df_all_reports["class"].isin(real_classs)]
metrics = ["precision", "recall", "f1-score"]

mean_std_df = (
    df_classs_only
    .groupby("class")[metrics]
    .agg(["mean", "std"])
    .reset_index()
)

mean_std_df.columns = [
    "class",
    "precision_mean", "precision_std",
    "recall_mean",    "recall_std",
    "f1_mean",        "f1_std"
]

print("\n" + "#" * 80)
print("PER-CLASS MEAN ± SAMPLE SD (outer tests)")
print("#" * 80)

for _, row in mean_std_df.iterrows():
    print(f"\nClass: {row['class']}")
    print(f"  Precision: {row['precision_mean']:.4f} ± {row['precision_std']:.4f}")
    print(f"  Recall:    {row['recall_mean']:.4f} ± {row['recall_std']:.4f}")
    print(f"  F1-score:  {row['f1_mean']:.4f} ± {row['f1_std']:.4f}")

## Historical experimental results
The following outputs are archival evidence extracted from the original notebook. They are separate from executable cells so that a new run cannot silently replace their provenance. Text labels are translated; numerical values are unchanged. Figure pixels are preserved, including the original Italian title on the pooled confusion matrix.

| Metric | Outer-fold mean ± sample SD |
|---|---|
| Macro precision | 0.8364 ± 0.0134 |
| Macro recall | 0.8678 ± 0.0168 |
| Macro F1 | 0.8465 ± 0.0173 |

Pooled outer-test accuracy: **0.8406** across 9,961 predictions. Pooled macro precision/recall/F1: **0.8314 / 0.8678 / 0.8463**. All three selected learning rate 3e-4 and batch size 16.


### Saved output from source cell 20
```text

================================================================================
OUTER FOLD A
================================================================================
params={'lr': 0.0003, 'batch': 16} -> mean inner MacroF1=0.7527
params={'lr': 0.0001, 'batch': 16} -> mean inner MacroF1=0.7042
params={'lr': 0.0003, 'batch': 32} -> mean inner MacroF1=0.6498
params={'lr': 0.0001, 'batch': 32} -> mean inner MacroF1=0.5562
>> BEST OUTER A: {'lr': 0.0003, 'batch': 16} (mean inner MacroF1=0.7527)
Epoch 1/25
332/332 - 19s - 56ms/step - accuracy: 0.5883 - loss: 0.8245 - val_accuracy: 0.3630 - val_loss: 1.8045 - learning_rate: 3.0000e-04
Epoch 2/25
332/332 - 11s - 33ms/step - accuracy: 0.6649 - loss: 0.6401 - val_accuracy: 0.7380 - val_loss: 0.5801 - learning_rate: 3.0000e-04
Epoch 3/25
332/332 - 11s - 33ms/step - accuracy: 0.6915 - loss: 0.5826 - val_accuracy: 0.6913 - val_loss: 0.6013 - learning_rate: 3.0000e-04
Epoch 4/25

Epoch 4: ReduceLROnPlateau reducing learning rate to 0.0001500000071246177.
332/332 - 11s - 33ms/step - accuracy: 0.7257 - loss: 0.5363 - val_accuracy: 0.4646 - val_loss: 1.9986 - learning_rate: 3.0000e-04
Epoch 5/25
332/332 - 11s - 33ms/step - accuracy: 0.7432 - loss: 0.4951 - val_accuracy: 0.7613 - val_loss: 0.5588 - learning_rate: 1.5000e-04
Epoch 6/25
332/332 - 11s - 33ms/step - accuracy: 0.7549 - loss: 0.4802 - val_accuracy: 0.8080 - val_loss: 0.4709 - learning_rate: 1.5000e-04
Epoch 7/25
332/332 - 11s - 33ms/step - accuracy: 0.7645 - loss: 0.4662 - val_accuracy: 0.4443 - val_loss: 1.4885 - learning_rate: 1.5000e-04
Epoch 8/25

Epoch 8: ReduceLROnPlateau reducing learning rate to 7.500000356230885e-05.
332/332 - 11s - 33ms/step - accuracy: 0.7758 - loss: 0.4508 - val_accuracy: 0.6190 - val_loss: 0.9061 - learning_rate: 1.5000e-04
Epoch 9/25
332/332 - 11s - 33ms/step - accuracy: 0.7869 - loss: 0.4223 - val_accuracy: 0.7327 - val_loss: 0.6151 - learning_rate: 7.5000e-05
Epoch 10/25

Epoch 10: ReduceLROnPlateau reducing learning rate to 3.7500001781154424e-05.
332/332 - 11s - 33ms/step - accuracy: 0.8033 - loss: 0.3973 - val_accuracy: 0.7628 - val_loss: 0.5296 - learning_rate: 7.5000e-05
Epoch 11/25
332/332 - 11s - 33ms/step - accuracy: 0.8099 - loss: 0.3806 - val_accuracy: 0.5791 - val_loss: 1.0351 - learning_rate: 3.7500e-05
Epoch 11: early stopping
Restoring model weights from the end of the best epoch: 6.

OUTER TEST A: MacroP=0.8269 MacroR=0.8490 MacroF1=0.8293

CLASSIFICATION REPORT (OUTER TEST):
                 precision    recall  f1-score   support

          COVID     0.7142    0.9096    0.8001      1206
         Normal     0.9160    0.7131    0.8019      1666
Viral Pneumonia     0.8504    0.9243    0.8858       449

       accuracy                         0.8130      3321
      macro avg     0.8269    0.8490    0.8293      3321
   weighted avg     0.8338    0.8130    0.8126      3321


```

![Historical figure from source cell 20](assets/results/source_cell_20_output_1.png)

### Saved output from source cell 20
```text

================================================================================
OUTER FOLD B
================================================================================
params={'lr': 0.0003, 'batch': 16} -> mean inner MacroF1=0.7516
params={'lr': 0.0001, 'batch': 16} -> mean inner MacroF1=0.7123
params={'lr': 0.0003, 'batch': 32} -> mean inner MacroF1=0.6446
params={'lr': 0.0001, 'batch': 32} -> mean inner MacroF1=0.5705
>> BEST OUTER B: {'lr': 0.0003, 'batch': 16} (mean inner MacroF1=0.7516)
Epoch 1/25
332/332 - 19s - 57ms/step - accuracy: 0.6037 - loss: 0.7925 - val_accuracy: 0.1663 - val_loss: 2.6277 - learning_rate: 3.0000e-04
Epoch 2/25
332/332 - 11s - 32ms/step - accuracy: 0.6826 - loss: 0.6147 - val_accuracy: 0.3679 - val_loss: 1.7210 - learning_rate: 3.0000e-04
Epoch 3/25
332/332 - 11s - 33ms/step - accuracy: 0.7167 - loss: 0.5666 - val_accuracy: 0.7269 - val_loss: 0.6742 - learning_rate: 3.0000e-04
Epoch 4/25
332/332 - 11s - 33ms/step - accuracy: 0.7359 - loss: 0.5441 - val_accuracy: 0.6697 - val_loss: 0.6718 - learning_rate: 3.0000e-04
Epoch 5/25
332/332 - 11s - 32ms/step - accuracy: 0.7492 - loss: 0.4983 - val_accuracy: 0.5711 - val_loss: 1.1612 - learning_rate: 3.0000e-04
Epoch 6/25

Epoch 6: ReduceLROnPlateau reducing learning rate to 0.0001500000071246177.
332/332 - 11s - 32ms/step - accuracy: 0.7668 - loss: 0.4752 - val_accuracy: 0.5816 - val_loss: 1.2687 - learning_rate: 3.0000e-04
Epoch 7/25
332/332 - 11s - 32ms/step - accuracy: 0.7816 - loss: 0.4444 - val_accuracy: 0.4778 - val_loss: 1.3726 - learning_rate: 1.5000e-04
Epoch 8/25
332/332 - 11s - 33ms/step - accuracy: 0.7991 - loss: 0.4153 - val_accuracy: 0.7344 - val_loss: 0.5689 - learning_rate: 1.5000e-04
Epoch 9/25
332/332 - 11s - 33ms/step - accuracy: 0.8020 - loss: 0.4124 - val_accuracy: 0.4808 - val_loss: 1.2893 - learning_rate: 1.5000e-04
Epoch 10/25
332/332 - 11s - 33ms/step - accuracy: 0.8200 - loss: 0.3813 - val_accuracy: 0.7878 - val_loss: 0.4975 - learning_rate: 1.5000e-04
Epoch 11/25
332/332 - 11s - 33ms/step - accuracy: 0.8261 - loss: 0.3641 - val_accuracy: 0.8518 - val_loss: 0.3897 - learning_rate: 1.5000e-04
Epoch 12/25
332/332 - 11s - 32ms/step - accuracy: 0.8210 - loss: 0.3798 - val_accuracy: 0.7713 - val_loss: 0.5308 - learning_rate: 1.5000e-04
Epoch 13/25

Epoch 13: ReduceLROnPlateau reducing learning rate to 7.500000356230885e-05.
332/332 - 11s - 32ms/step - accuracy: 0.8289 - loss: 0.3559 - val_accuracy: 0.5907 - val_loss: 1.3734 - learning_rate: 1.5000e-04
Epoch 14/25
332/332 - 11s - 32ms/step - accuracy: 0.8451 - loss: 0.3267 - val_accuracy: 0.7682 - val_loss: 0.5466 - learning_rate: 7.5000e-05
Epoch 15/25

Epoch 15: ReduceLROnPlateau reducing learning rate to 3.7500001781154424e-05.
332/332 - 11s - 33ms/step - accuracy: 0.8447 - loss: 0.3331 - val_accuracy: 0.8164 - val_loss: 0.4759 - learning_rate: 7.5000e-05
Epoch 16/25
332/332 - 11s - 33ms/step - accuracy: 0.8560 - loss: 0.3050 - val_accuracy: 0.5305 - val_loss: 1.5228 - learning_rate: 3.7500e-05
Epoch 16: early stopping
Restoring model weights from the end of the best epoch: 11.

OUTER TEST B: MacroP=0.8305 MacroR=0.8733 MacroF1=0.8463

CLASSIFICATION REPORT (OUTER TEST):
                 precision    recall  f1-score   support

          COVID     0.8778    0.7925    0.8330      1205
         Normal     0.8697    0.8608    0.8652      1667
Viral Pneumonia     0.7440    0.9665    0.8408       448

       accuracy                         0.8503      3320
      macro avg     0.8305    0.8733    0.8463      3320
   weighted avg     0.8557    0.8503    0.8502      3320


```

![Historical figure from source cell 20](assets/results/source_cell_20_output_3.png)

### Saved output from source cell 20
```text

================================================================================
OUTER FOLD C
================================================================================
params={'lr': 0.0003, 'batch': 16} -> mean inner MacroF1=0.8067
params={'lr': 0.0001, 'batch': 16} -> mean inner MacroF1=0.7681
params={'lr': 0.0003, 'batch': 32} -> mean inner MacroF1=0.7129
params={'lr': 0.0001, 'batch': 32} -> mean inner MacroF1=0.5675
>> BEST OUTER C: {'lr': 0.0003, 'batch': 16} (mean inner MacroF1=0.8067)
Epoch 1/25
332/332 - 19s - 57ms/step - accuracy: 0.5932 - loss: 0.8148 - val_accuracy: 0.2295 - val_loss: 2.5417 - learning_rate: 3.0000e-04
Epoch 2/25
332/332 - 11s - 32ms/step - accuracy: 0.6749 - loss: 0.6265 - val_accuracy: 0.5591 - val_loss: 0.9489 - learning_rate: 3.0000e-04
Epoch 3/25
332/332 - 11s - 33ms/step - accuracy: 0.6980 - loss: 0.5783 - val_accuracy: 0.6321 - val_loss: 0.7345 - learning_rate: 3.0000e-04
Epoch 4/25
332/332 - 11s - 32ms/step - accuracy: 0.7227 - loss: 0.5455 - val_accuracy: 0.6095 - val_loss: 0.8379 - learning_rate: 3.0000e-04
Epoch 5/25

Epoch 5: ReduceLROnPlateau reducing learning rate to 0.0001500000071246177.
332/332 - 11s - 32ms/step - accuracy: 0.7387 - loss: 0.5127 - val_accuracy: 0.6305 - val_loss: 0.8553 - learning_rate: 3.0000e-04
Epoch 6/25
332/332 - 11s - 32ms/step - accuracy: 0.7679 - loss: 0.4548 - val_accuracy: 0.8412 - val_loss: 0.3966 - learning_rate: 1.5000e-04
Epoch 7/25
332/332 - 11s - 33ms/step - accuracy: 0.7788 - loss: 0.4454 - val_accuracy: 0.5199 - val_loss: 1.3661 - learning_rate: 1.5000e-04
Epoch 8/25

Epoch 8: ReduceLROnPlateau reducing learning rate to 7.500000356230885e-05.
332/332 - 11s - 32ms/step - accuracy: 0.7841 - loss: 0.4324 - val_accuracy: 0.5335 - val_loss: 1.3921 - learning_rate: 1.5000e-04
Epoch 9/25
332/332 - 11s - 33ms/step - accuracy: 0.7974 - loss: 0.4023 - val_accuracy: 0.8555 - val_loss: 0.3538 - learning_rate: 7.5000e-05
Epoch 10/25
332/332 - 11s - 32ms/step - accuracy: 0.8093 - loss: 0.3872 - val_accuracy: 0.8555 - val_loss: 0.3509 - learning_rate: 7.5000e-05
Epoch 11/25
332/332 - 11s - 32ms/step - accuracy: 0.8149 - loss: 0.3851 - val_accuracy: 0.7976 - val_loss: 0.4466 - learning_rate: 7.5000e-05
Epoch 12/25

Epoch 12: ReduceLROnPlateau reducing learning rate to 3.7500001781154424e-05.
332/332 - 11s - 32ms/step - accuracy: 0.8176 - loss: 0.3783 - val_accuracy: 0.8126 - val_loss: 0.4219 - learning_rate: 7.5000e-05
Epoch 13/25
332/332 - 11s - 32ms/step - accuracy: 0.8225 - loss: 0.3600 - val_accuracy: 0.6524 - val_loss: 0.7781 - learning_rate: 3.7500e-05
Epoch 14/25

Epoch 14: ReduceLROnPlateau reducing learning rate to 1.8750000890577212e-05.
332/332 - 11s - 32ms/step - accuracy: 0.8309 - loss: 0.3562 - val_accuracy: 0.6764 - val_loss: 0.7535 - learning_rate: 3.7500e-05
Epoch 15/25
332/332 - 11s - 32ms/step - accuracy: 0.8334 - loss: 0.3453 - val_accuracy: 0.7201 - val_loss: 0.6366 - learning_rate: 1.8750e-05
Epoch 15: early stopping
Restoring model weights from the end of the best epoch: 10.

OUTER TEST C: MacroP=0.8517 MacroR=0.8811 MacroF1=0.8640

CLASSIFICATION REPORT (OUTER TEST):
                 precision    recall  f1-score   support

          COVID     0.8062    0.8838    0.8432      1205
         Normal     0.9117    0.8176    0.8621      1667
Viral Pneumonia     0.8373    0.9420    0.8866       448

       accuracy                         0.8584      3320
      macro avg     0.8517    0.8811    0.8640      3320
   weighted avg     0.8634    0.8584    0.8586      3320


```

![Historical figure from source cell 20](assets/results/source_cell_20_output_5.png)

### Saved output from source cell 22
```text
Available histories: ['A', 'B', 'C']

```

![Historical figure from source cell 22](assets/results/source_cell_22_output_1.png)

![Historical figure from source cell 22](assets/results/source_cell_22_output_2.png)

![Historical figure from source cell 22](assets/results/source_cell_22_output_3.png)

### Saved output from source cell 25
```text
Total pooled outer-test samples: 9961

POOLED CLASSIFICATION REPORT (A+B+C):
                 precision    recall  f1-score   support

          COVID     0.7901    0.8620    0.8245      3616
         Normal     0.8973    0.7972    0.8443      5000
Viral Pneumonia     0.8069    0.9442    0.8702      1345

       accuracy                         0.8406      9961
      macro avg     0.8314    0.8678    0.8463      9961
   weighted avg     0.8462    0.8406    0.8406      9961


```

![Historical figure from source cell 25](assets/results/source_cell_25_output_1.png)

### Saved output from source cell 28
```text
        Macro Precision  Macro Recall  Macro F1
Fold A         0.826854      0.848993  0.829281
Fold B         0.830480      0.873292  0.846329
Fold C         0.851738      0.881139  0.863966
```

### Saved output from source cell 31
```text

################################################################################
FINAL RESULTS — NESTED 3×3 (outer-test mean ± sample SD)
################################################################################
Macro Precision: 0.8364 ± 0.0134
Macro Recall:    0.8678 ± 0.0168
Macro F1:        0.8465 ± 0.0173

Best params per fold: [('A', {'lr': 0.0003, 'batch': 16}, 0.7526703348206069), ('B', {'lr': 0.0003, 'batch': 16}, 0.7516101467793407), ('C', {'lr': 0.0003, 'batch': 16}, 0.8067457458388021)]

```

### Saved output from source cell 34
```text

################################################################################
PER-CLASS MEAN ± SAMPLE SD (outer tests)
################################################################################

Class: COVID
  Precision: 0.7994 ± 0.0820
  Recall:    0.8620 ± 0.0615
  F1-score:  0.8254 ± 0.0225

Class: Normal
  Precision: 0.8991 ± 0.0256
  Recall:    0.7972 ± 0.0760
  F1-score:  0.8431 ± 0.0357

Class: Viral Pneumonia
  Precision: 0.8106 ± 0.0580
  Recall:    0.9443 ± 0.0212
  F1-score:  0.8710 ± 0.0262

```

## Discussion and limitations
The outer-test partitions are separated by image indices from training and tuning. Patient identifiers, content hashes and the original image files are unavailable for this audit; duplicate images and cross-patient overlap could not be checked. This is image-level stratification, not demonstrated patient-independent validation. No external institution or prospective validation is provided, and the model has not been validated for clinical use.

Normal has pooled recall 0.7972, compared with COVID 0.8620 and Viral Pneumonia 0.9442. The pooled F1 (0.8463) exceeds pooled accuracy (0.8406); the original contrary interpretation has been removed. Three folds cannot establish robustness to acquisition site or patient population. Class and acquisition-source confounding remains unassessed. The 7-epoch inner budget and 25-epoch final budget are preserved and may select hyperparameters differently from a longer search.

## Conclusions
The saved experiment documents a compact CNN evaluated with nested image-level cross-validation. Its results describe the original local subset and protocol; they do not establish clinical utility or external generalisation. See VALIDATION.md for changes and checks.
